In [16]:

!pip -q install requests owslib rasterio geopandas shapely pyproj matplotlib pandas

import os
import json
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from owslib.wcs import WebCoverageService
from pyproj import Transformer
from shapely.geometry import mapping
from shapely.ops import transform as transform_geometry
from rasterio.features import geometry_mask
from IPython.display import display

# ==================================================
# 1. SETUP AND LOAD AOI
# ==================================================

AOI_PATH = "/content/El_Zankalon_AOI.geojson"
OUTPUT_DIR = "/content/soilgrids_output"
ZIP_PATH = "/content/SoilGrids_El_Zankalon.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(AOI_PATH):
    raise FileNotFoundError(
        f"Upload the AOI file first: {AOI_PATH}"
    )

aoi_gdf = gpd.read_file(AOI_PATH)

if aoi_gdf.empty or aoi_gdf.geometry.is_empty.all():
    raise ValueError("AOI GeoJSON is empty.")

if aoi_gdf.crs is None:
    aoi_gdf = aoi_gdf.set_crs("EPSG:4326")

aoi_gdf = aoi_gdf.to_crs("EPSG:4326")
aoi = aoi_gdf.geometry.union_all()

if not aoi.is_valid or aoi.is_empty:
    raise ValueError("Invalid AOI geometry.")

west, south, east, north = aoi.bounds

print("AOI loaded successfully!")
print("Bounds:", aoi.bounds)

SOILGRIDS_CRS = "+proj=igh +datum=WGS84 +no_defs +towgs84=0,0,0"

aoi_transformer = Transformer.from_crs(
    "EPSG:4326",
    SOILGRIDS_CRS,
    always_xy=True
)

projected_aoi = transform_geometry(
    aoi_transformer.transform,
    aoi
)

projected_bbox = projected_aoi.bounds

print("Projected BBOX:", projected_bbox)

# ==================================================
# 2. SOILGRIDS PROPERTIES
# ==================================================

properties = {
    "ph": {
        "service": "phh2o",
        "url": "https://maps.isric.org/mapserv?map=/map/phh2o.map",
        "factor": 10,
        "unit": "pH",
        "label": "Soil pH"
    },
    "soc": {
        "service": "soc",
        "url": "https://maps.isric.org/mapserv?map=/map/soc.map",
        "factor": 10,
        "unit": "g/kg",
        "label": "Soil Organic Carbon"
    },
    "clay": {
        "service": "clay",
        "url": "https://maps.isric.org/mapserv?map=/map/clay.map",
        "factor": 10,
        "unit": "%",
        "label": "Clay"
    },
    "sand": {
        "service": "sand",
        "url": "https://maps.isric.org/mapserv?map=/map/sand.map",
        "factor": 10,
        "unit": "%",
        "label": "Sand"
    },
    "silt": {
        "service": "silt",
        "url": "https://maps.isric.org/mapserv?map=/map/silt.map",
        "factor": 10,
        "unit": "%",
        "label": "Silt"
    },
    "cec": {
        "service": "cec",
        "url": "https://maps.isric.org/mapserv?map=/map/cec.map",
        "factor": 10,
        "unit": "cmol(c)/kg",
        "label": "CEC"
    },
    "nitrogen": {
        "service": "nitrogen",
        "url": "https://maps.isric.org/mapserv?map=/map/nitrogen.map",
        "factor": 100,
        "unit": "g/kg",
        "label": "Nitrogen"
    },
    "bdod": {
        "service": "bdod",
        "url": "https://maps.isric.org/mapserv?map=/map/bdod.map",
        "factor": 100,
        "unit": "kg/dm3",
        "label": "Bulk Density"
    }
}

# ==================================================
# 3. DOWNLOAD AND PROCESS MEDIAN Q0.5
# ==================================================

cleaned_data = {}
summary_rows = []
reference = None
raster_metadata = {}

for name, info in properties.items():

    print(f"\nDownloading {name.upper()}...")

    wcs = WebCoverageService(
        info["url"],
        version="1.0.0"
    )

    coverage_id = f"{info['service']}_0-5cm_Q0.5"

    if coverage_id not in wcs.contents:
        raise ValueError(
            f"Coverage not available: {coverage_id}"
        )

    response = wcs.getCoverage(
        identifier=coverage_id,
        bbox=projected_bbox,
        crs="urn:ogc:def:crs:EPSG::152160",
        resx=250,
        resy=250,
        format="GEOTIFF_INT16",
        timeout=180
    )

    file_bytes = response.read()

    if file_bytes[:4] not in [b"II*\x00", b"MM\x00*"]:
        raise RuntimeError(
            f"Invalid GeoTIFF response for {name}:\n"
            + file_bytes[:1000].decode(
                "utf-8",
                errors="replace"
            )
        )

    raw_file = os.path.join(
        OUTPUT_DIR,
        f"soilgrids_{name}_Q0.5_raw.tif"
    )

    with open(raw_file, "wb") as f:
        f.write(file_bytes)

    with rasterio.open(raw_file) as src:

        raw = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
        raster_crs = src.crs
        raster_transform = src.transform
        raster_shape = src.shape

        if reference is None:
            reference = {
                "shape": raster_shape,
                "crs": raster_crs,
                "transform": raster_transform
            }

        else:
            if raster_shape != reference["shape"]:
                raise ValueError(
                    f"{name}: Shape mismatch!"
                )

            if raster_crs != reference["crs"]:
                raise ValueError(
                    f"{name}: CRS mismatch!"
                )

            if not np.allclose(
                tuple(raster_transform),
                tuple(reference["transform"]),
                rtol=0,
                atol=1e-6
            ):
                raise ValueError(
                    f"{name}: Pixel alignment mismatch!"
                )

    # Transform AOI into actual raster CRS
    raster_transformer = Transformer.from_crs(
        "EPSG:4326",
        raster_crs,
        always_xy=True
    )

    aoi_in_raster_crs = transform_geometry(
        raster_transformer.transform,
        aoi
    )

    inside_aoi = geometry_mask(
        [mapping(aoi_in_raster_crs)],
        out_shape=raw.shape,
        transform=raster_transform,
        invert=True
    )

    # Handle NoData
    if nodata is not None:
        raw[raw == nodata] = np.nan

    raw[~inside_aoi] = np.nan

    # Convert to physical units
    clean = raw / info["factor"]

    valid = clean[np.isfinite(clean)]

    if valid.size == 0:
        raise ValueError(
            f"{name}: No valid pixels inside AOI!"
        )

    # Save cleaned raster
    output_profile = profile.copy()

    output_profile.update(
        driver="GTiff",
        dtype="float32",
        nodata=-9999,
        compress="lzw"
    )

    clean_file = os.path.join(
        OUTPUT_DIR,
        f"soilgrids_{name}_Q0.5_clean.tif"
    )

    output_array = np.where(
        np.isfinite(clean),
        clean,
        -9999
    ).astype("float32")

    with rasterio.open(
        clean_file,
        "w",
        **output_profile
    ) as dst:
        dst.write(output_array, 1)

    cleaned_data[name] = clean

    raster_metadata[name] = {
        "shape": list(raster_shape),
        "crs": str(raster_crs),
        "transform": list(raster_transform)
    }

    summary_rows.append({
        "Property": name.upper(),
        "Statistic": "Q0.5 Median",
        "Depth": "0-5 cm",
        "Unit": info["unit"],
        "Mean": float(np.mean(valid)),
        "Min": float(np.min(valid)),
        "Max": float(np.max(valid)),
        "Std": float(np.std(valid)),
        "Valid_Pixels": int(valid.size),
        "Missing_Pixels": int(
            clean.size - valid.size
        )
    })

    # Save map
    plt.figure(figsize=(8, 6))
    plt.imshow(clean, cmap="viridis")
    plt.colorbar(label=info["unit"])
    plt.title(
        f"{info['label']} - El Zankalon (Q0.5)"
    )
    plt.xlabel("Columns")
    plt.ylabel("Rows")

    map_file = os.path.join(
        OUTPUT_DIR,
        f"soilgrids_{name}_Q0.5_map.png"
    )

    plt.savefig(
        map_file,
        dpi=200,
        bbox_inches="tight"
    )
    plt.close()

    print(
        f"{name.upper()} completed | "
        f"Valid pixels: {valid.size} | "
        f"Mean: {np.mean(valid):.4f}"
    )

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "soilgrids_Q0.5_summary.csv"
    ),
    index=False
)

print("\nMEDIAN SUMMARY")
display(summary_df)

# ==================================================
# 4. DOWNLOAD MEAN TEXTURE LAYERS
# ==================================================

mean_texture = {}
mean_summary_rows = []

for name in ["clay", "sand", "silt"]:

    info = properties[name]

    print(f"\nDownloading {name.upper()} MEAN...")

    wcs = WebCoverageService(
        info["url"],
        version="1.0.0"
    )

    coverage_id = f"{name}_0-5cm_mean"

    if coverage_id not in wcs.contents:
        raise ValueError(
            f"Mean coverage not available: {coverage_id}"
        )

    response = wcs.getCoverage(
        identifier=coverage_id,
        bbox=projected_bbox,
        crs="urn:ogc:def:crs:EPSG::152160",
        resx=250,
        resy=250,
        format="GEOTIFF_INT16",
        timeout=180
    )

    file_bytes = response.read()

    if file_bytes[:4] not in [b"II*\x00", b"MM\x00*"]:
        raise RuntimeError(
            f"Invalid Mean GeoTIFF response for {name}"
        )

    raw_file = os.path.join(
        OUTPUT_DIR,
        f"soilgrids_{name}_mean_raw.tif"
    )

    with open(raw_file, "wb") as f:
        f.write(file_bytes)

    with rasterio.open(raw_file) as src:

        values = src.read(1).astype("float32")
        nodata = src.nodata
        profile = src.profile.copy()
        raster_crs = src.crs
        raster_transform = src.transform

        if src.shape != reference["shape"]:
            raise ValueError(
                f"{name} Mean: Shape mismatch!"
            )

        if raster_crs != reference["crs"]:
            raise ValueError(
                f"{name} Mean: CRS mismatch!"
            )

        if not np.allclose(
            tuple(raster_transform),
            tuple(reference["transform"]),
            rtol=0,
            atol=1e-6
        ):
            raise ValueError(
                f"{name} Mean: Pixel alignment mismatch!"
            )

    if nodata is not None:
        values[values == nodata] = np.nan

    raster_transformer = Transformer.from_crs(
        "EPSG:4326",
        raster_crs,
        always_xy=True
    )

    aoi_in_raster_crs = transform_geometry(
        raster_transformer.transform,
        aoi
    )

    inside_aoi = geometry_mask(
        [mapping(aoi_in_raster_crs)],
        out_shape=values.shape,
        transform=raster_transform,
        invert=True
    )

    values[~inside_aoi] = np.nan

    clean_mean = values / info["factor"]

    valid = clean_mean[np.isfinite(clean_mean)]

    if valid.size == 0:
        raise ValueError(
            f"{name} Mean: No valid pixels!"
        )

    mean_texture[name] = clean_mean

    output_profile = profile.copy()

    output_profile.update(
        driver="GTiff",
        dtype="float32",
        nodata=-9999,
        compress="lzw"
    )

    clean_file = os.path.join(
        OUTPUT_DIR,
        f"soilgrids_{name}_mean_clean.tif"
    )

    output_array = np.where(
        np.isfinite(clean_mean),
        clean_mean,
        -9999
    ).astype("float32")

    with rasterio.open(
        clean_file,
        "w",
        **output_profile
    ) as dst:
        dst.write(output_array, 1)

    mean_summary_rows.append({
        "Property": name.upper(),
        "Statistic": "Mean",
        "Depth": "0-5 cm",
        "Unit": info["unit"],
        "Mean": float(np.mean(valid)),
        "Min": float(np.min(valid)),
        "Max": float(np.max(valid)),
        "Std": float(np.std(valid)),
        "Valid_Pixels": int(valid.size),
        "Missing_Pixels": int(
            clean_mean.size - valid.size
        )
    })

    print(
        f"{name.upper()} Mean completed | "
        f"Valid pixels: {valid.size}"
    )

mean_summary_df = pd.DataFrame(
    mean_summary_rows
)

mean_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "soilgrids_mean_texture_summary.csv"
    ),
    index=False
)

print("\nMEAN TEXTURE SUMMARY")
display(mean_summary_df)

# ==================================================
# 5. RASTER QUALITY CHECK
# ==================================================

print("\n========== RASTER QUALITY CHECK ==========")

for name in properties:

    clean_file = os.path.join(
        OUTPUT_DIR,
        f"soilgrids_{name}_Q0.5_clean.tif"
    )

    with rasterio.open(clean_file) as src:

        valid_count = src.read(
            1,
            masked=True
        ).count()

        assert src.shape == reference["shape"]
        assert src.crs == reference["crs"]

        assert np.allclose(
            tuple(src.transform),
            tuple(reference["transform"]),
            rtol=0,
            atol=1e-6
        )

        print(
            name.upper(),
            "| Shape:", src.shape,
            "| Valid:", valid_count
        )

# ==================================================
# 6. TEXTURE CONSISTENCY CHECK
# ==================================================

def texture_check(texture_dict, label):

    stack = np.stack([
        texture_dict["clay"],
        texture_dict["sand"],
        texture_dict["silt"]
    ])

    valid_mask = np.all(
        np.isfinite(stack),
        axis=0
    )

    texture_sum = np.sum(
        stack,
        axis=0
    )

    valid_values = texture_sum[valid_mask]

    if valid_values.size == 0:
        raise ValueError(
            f"No common valid pixels for {label}"
        )

    result = {
        "Statistic": label,
        "Mean_Texture_Sum": float(
            np.mean(valid_values)
        ),
        "Min_Texture_Sum": float(
            np.min(valid_values)
        ),
        "Max_Texture_Sum": float(
            np.max(valid_values)
        ),
        "Pixels_95_to_105": int(
            np.sum(
                (valid_values >= 95) &
                (valid_values <= 105)
            )
        ),
        "Valid_Texture_Pixels": int(
            valid_values.size
        )
    }

    print(f"\n========== {label} TEXTURE CHECK ==========")

    for key, value in result.items():
        print(f"{key}: {value}")

    return result


median_texture_result = texture_check(
    cleaned_data,
    "Q0.5 MEDIAN"
)

mean_texture_result = texture_check(
    mean_texture,
    "MEAN"
)

texture_results_df = pd.DataFrame([
    median_texture_result,
    mean_texture_result
])

texture_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "soilgrids_texture_checks.csv"
    ),
    index=False
)

display(texture_results_df)

# ==================================================
# 7. SAVE METADATA
# ==================================================

metadata = {
    "Source": "ISRIC SoilGrids 2.0",
    "Study_Area": "El Zankalon AOI, Sharqia, Egypt",
    "AOI_File": AOI_PATH,
    "AOI_Geometry_Type": aoi.geom_type,
    "AOI_WGS84_Bounds": {
        "west": float(west),
        "south": float(south),
        "east": float(east),
        "north": float(north)
    },
    "Depth": "0-5 cm",
    "Median_Quantile": "Q0.5",
    "Target_Resolution": "250 m",
    "Median_Properties": list(properties.keys()),
    "Mean_Properties": [
        "clay",
        "sand",
        "silt"
    ],
    "Note": (
        "SoilGrids contains spatial predictions, "
        "not daily observations. Mean and Q0.5 "
        "are separate statistical layers."
    ),
    "Raster_Metadata": raster_metadata
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "soilgrids_metadata.json"
    ),
    "w"
) as f:
    json.dump(metadata, f, indent=4)

# ==================================================
# 8. CREATE FINAL ZIP
# ==================================================

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files_list in os.walk(
        OUTPUT_DIR
    ):

        for filename in files_list:

            filepath = os.path.join(
                root,
                filename
            )

            arcname = os.path.relpath(
                filepath,
                OUTPUT_DIR
            )

            zipf.write(
                filepath,
                arcname
            )

print("\n========== FINAL RESULT ==========")
print("Raster alignment checks completed.")
print("Texture diagnostics completed.")
print("ZIP created:", ZIP_PATH)
print("Review texture checks before final approval.")

from google.colab import files
files.download(ZIP_PATH)


AOI loaded successfully!
Bounds: (31.35, 30.44, 31.55, 30.64)
Projected BBOX: (3468884.727218316, 3388565.2997472477, 3488346.333882893, 3410829.1979059023)



/tmp/ipykernel_1113/3978071366.py:61: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  projected_aoi = transform_geometry(
/tmp/ipykernel_1113/3978071366.py:231: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  aoi_in_raster_crs = transform_geometry(


PH completed | Valid pixels: 6128 | Mean: 7.6490



/tmp/ipykernel_1113/3978071366.py:231: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  aoi_in_raster_crs = transform_geometry(


SOC completed | Valid pixels: 6128 | Mean: 11.6685



/tmp/ipykernel_1113/3978071366.py:231: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  aoi_in_raster_crs = transform_geometry(


... (19 outputs left out)

,Statistic,Mean_Texture_Sum,Min_Texture_Sum,Max_Texture_Sum,Pixels_95_to_105,Valid_Texture_Pixels
0,Q0.5 MEDIAN,76.908165,70.099998,84.099991,0,6128
1,MEAN,100.000961,99.899994,100.100006,6128,6128



========== FINAL RESULT ==========
Raster alignment checks completed.
Texture diagnostics completed.
ZIP created: /content/SoilGrids_El_Zankalon.zip
Review texture checks before final approval.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import os
import glob
import numpy as np
import rasterio

# Find the saved Q0.5 raster files
for f in glob.glob("/content/**/*.tif", recursive=True):
    if any(x in os.path.basename(f).lower() for x in ["clay", "sand", "silt"]):
        print(f)

/content/soilgrids_output/soilgrids_sand_mean_raw.tif
/content/soilgrids_output/soilgrids_silt_mean_clean.tif
/content/soilgrids_output/soilgrids_sand_Q0.5_clean.tif
/content/soilgrids_output/soilgrids_clay_Q0.5_raw.tif
/content/soilgrids_output/soilgrids_sand_Q0.5_raw.tif
/content/soilgrids_output/soilgrids_sand_mean_clean.tif
/content/soilgrids_output/soilgrids_silt_mean_raw.tif
/content/soilgrids_output/soilgrids_clay_mean_raw.tif
/content/soilgrids_output/soilgrids_clay_mean_clean.tif
/content/soilgrids_output/soilgrids_silt_Q0.5_clean.tif
/content/soilgrids_output/soilgrids_silt_Q0.5_raw.tif
/content/soilgrids_output/soilgrids_clay_Q0.5_clean.tif


In [19]:
import os
import numpy as np
import rasterio

folder = "/content/soilgrids_output"

files = {
    "clay": os.path.join(folder, "soilgrids_clay_Q0.5_clean.tif"),
    "sand": os.path.join(folder, "soilgrids_sand_Q0.5_clean.tif"),
    "silt": os.path.join(folder, "soilgrids_silt_Q0.5_clean.tif")
}

data = {}
profile = None

for name, path in files.items():
    with rasterio.open(path) as src:
        data[name] = src.read(1).astype("float32")
        if profile is None:
            profile = src.profile.copy()

clay = data["clay"]
sand = data["sand"]
silt = data["silt"]

# Check raster alignment
assert clay.shape == sand.shape == silt.shape

# Identify valid pixels
valid = (
    np.isfinite(clay) &
    np.isfinite(sand) &
    np.isfinite(silt) &
    (clay >= 0) & (sand >= 0) & (silt >= 0)
)

total = clay + sand + silt
valid &= total > 0

# Normalize
normalized = {}

for name, values in data.items():
    result = np.full(values.shape, -9999, dtype="float32")
    result[valid] = values[valid] / total[valid] * 100
    normalized[name] = result

# Save normalized rasters
profile.update(dtype="float32", nodata=-9999, count=1)

for name, values in normalized.items():
    output_path = os.path.join(
        folder, f"soilgrids_{name}_Q0.5_normalized.tif"
    )

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(values, 1)

# Verify results
texture_sum = (
    normalized["clay"][valid] +
    normalized["sand"][valid] +
    normalized["silt"][valid]
)

print("Valid pixels:", int(valid.sum()))
print("Mean normalized sum:", float(np.mean(texture_sum)))
print("Minimum sum:", float(np.min(texture_sum)))
print("Maximum sum:", float(np.max(texture_sum)))
print("Pixels between 99.9 and 100.1%:",
      int(np.sum((texture_sum >= 99.9) & (texture_sum <= 100.1))))

print("\nNormalized files saved successfully!")

Valid pixels: 6128
Mean normalized sum: 100.0
Minimum sum: 99.99998474121094
Maximum sum: 100.00001525878906
Pixels between 99.9 and 100.1%: 6128

Normalized files saved successfully!


In [20]:
import os
import numpy as np
import rasterio

folder = "/content/soilgrids_output"

names = ["clay", "sand", "silt"]
arrays = {}

for name in names:
    path = os.path.join(
        folder, f"soilgrids_{name}_Q0.5_normalized.tif"
    )

    with rasterio.open(path) as src:
        arrays[name] = src.read(1)
        print(
            name.upper(),
            "| Shape:", src.shape,
            "| CRS:", src.crs,
            "| File exists:", os.path.exists(path)
        )

valid = np.ones(arrays["clay"].shape, dtype=bool)

for name in names:
    valid &= np.isfinite(arrays[name]) & (arrays[name] != -9999)

texture_sum = (
    arrays["clay"][valid] +
    arrays["sand"][valid] +
    arrays["silt"][valid]
)

print("\nValid pixels:", valid.sum())
print("Average sum:", texture_sum.mean())
print("Minimum:", texture_sum.min())
print("Maximum:", texture_sum.max())

CLAY | Shape: (89, 78) | CRS: PROJCS["unknown",GEOGCS["unknown",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Interrupted_Goode_Homolosine"],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] | File exists: True
SAND | Shape: (89, 78) | CRS: PROJCS["unknown",GEOGCS["unknown",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Interrupted_Goode_Homolosine"],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] | File exists: True
SILT | Shape: (89, 78) | CRS: PROJCS["unknown",GEOGCS["unkno

In [22]:
import os
import pandas as pd

files = [
    "/content/FarmWise_Unified_Dataset.csv",
    "/content/FarmWise_Data_Dictionary.csv"
]

for path in files:
    print("\n" + "="*60)
    print("FILE:", os.path.basename(path))

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:")
        print(df.columns.tolist())

        if "date" in df.columns:
            print("Date range:", df["date"].min(), "to", df["date"].max())

        display(df.head(3))
    else:
        print("File not found!")


FILE: FarmWise_Unified_Dataset.csv
Shape: (4018, 61)
Columns:
['date', 'year', 'month', 'day_of_year', 'aoi_center_lat', 'aoi_center_lon', 'power_t2m_mean_c', 'power_t2m_max_c', 'power_t2m_min_c', 'power_precip_mm_day', 'power_rh2m_pct', 'power_wind2m_ms', 'power_solar_rad_mj_m2_day', 'gpm_precip_mm_day', 'smap_soil_moisture_am_m3m3', 'smap_at_min_value_flag', 'ndvi_composite_date', 'ndvi_mean', 'ndvi_median', 'ndvi_std', 'ndvi_n_pixels', 'ndvi_share_usefulness_code2', 'ndvi_pixel_coverage_frac', 'ndvi_low_coverage_flag', 'ndvi_days_since_composite_start', 'soil_ph', 'soil_soc_g_kg', 'soil_clay_pct', 'soil_sand_pct', 'soil_silt_pct', 'soil_cec_cmolc_kg', 'soil_nitrogen_g_kg', 'soil_bdod_kg_dm3', 'gaez_cow_hilm_suitable_pct', 'gaez_cow_hrlm_suitable_pct', 'gaez_cow_lilm_suitable_pct', 'gaez_cow_lrlm_suitable_pct', 'gaez_mze_hilm_suitable_pct', 'gaez_mze_hrlm_suitable_pct', 'gaez_mze_lilm_suitable_pct', 'gaez_mze_lrlm_suitable_pct', 'gaez_pea_hilm_suitable_pct', 'gaez_pea_hrlm_suitable_

,date,year,month,day_of_year,aoi_center_lat,aoi_center_lon,power_t2m_mean_c,power_t2m_max_c,power_t2m_min_c,power_precip_mm_day,...,gaez_rcd_lilm_suitable_pct,gaez_rcd_lrlm_suitable_pct,gaez_rcw_hilm_suitable_pct,gaez_rcw_hrlm_suitable_pct,gaez_rcw_lilm_suitable_pct,gaez_rcw_lrlm_suitable_pct,gaez_whe_hilm_suitable_pct,gaez_whe_hrlm_suitable_pct,gaez_whe_lilm_suitable_pct,gaez_whe_lrlm_suitable_pct
0,2015-01-01,2015,1,1,30.54,31.45,10.78,16.83,6.40,0.00,...,0.0,0.0,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0
1,2015-01-02,2015,1,2,30.54,31.45,11.23,19.55,4.52,0.00,...,0.0,0.0,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0
2,2015-01-03,2015,1,3,30.54,31.45,11.16,18.54,5.19,0.19,...,0.0,0.0,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0



FILE: FarmWise_Data_Dictionary.csv
Shape: (61, 6)
Columns:
['column', 'source', 'unit', 'description', 'processing', 'raw_source_missing_pct']


,column,source,unit,description,processing,raw_source_missing_pct
0,date,spine,YYYY-MM-DD,Calendar day (one row per day),Complete daily range of all sources,NaN
1,year,derived,-,Year of date,From date,NaN
2,month,derived,-,Month of date,From date,NaN


In [23]:
import pandas as pd
import numpy as np

path = "/content/FarmWise_Unified_Dataset.csv"

df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"])
df["ndvi_composite_date"] = pd.to_datetime(
    df["ndvi_composite_date"], errors="coerce"
)

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nNDVI CHECK")
print("Rows with observed NDVI:", df["ndvi_mean"].notna().sum())
print("Unique composite dates:",
      df["ndvi_composite_date"].nunique())
print("Rows with composite date:",
      df["ndvi_composite_date"].notna().sum())

print("\nLOW COVERAGE")
print(df["ndvi_low_coverage_flag"].value_counts(dropna=False))

print("\nNDVI COMPOSITE SUMMARY")

composites = (
    df.dropna(subset=["ndvi_composite_date"])
      .groupby("ndvi_composite_date")
      .agg(
          ndvi_mean=("ndvi_mean", "mean"),
          pixel_coverage=("ndvi_pixel_coverage_frac", "mean"),
          observations=("ndvi_mean", "count"),
          low_coverage=("ndvi_low_coverage_flag", "max")
      )
      .reset_index()
)

print("Composite-level rows:", len(composites))
display(composites.head(10))

print("\nMISSING VALUES")
cols = [
    "power_t2m_mean_c",
    "power_precip_mm_day",
    "smap_soil_moisture_am_m3m3",
    "ndvi_mean",
    "soil_ph",
    "soil_clay_pct",
    "gaez_whe_hilm_suitable_pct"
]

display(df[cols].isna().mean().mul(100).round(2))

DATASET OVERVIEW
Shape: (4018, 61)
Date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00

NDVI CHECK
Rows with observed NDVI: 3941
Unique composite dates: 248
Rows with composite date: 3941

LOW COVERAGE
ndvi_low_coverage_flag
False    3781
True      160
NaN        77
Name: count, dtype: int64

NDVI COMPOSITE SUMMARY
Composite-level rows: 248


,ndvi_composite_date,ndvi_mean,pixel_coverage,observations,low_coverage
0,2015-01-01,0.687144,0.997984,16,False
1,2015-01-17,0.703984,0.982361,16,False
2,2015-02-02,0.697882,0.992944,16,False
3,2015-02-18,0.682736,0.966234,16,False
4,2015-03-06,0.682831,0.946201,16,False
5,2015-03-22,0.602478,0.999748,16,False
6,2015-04-07,0.518043,0.995464,16,False
7,2015-04-23,0.399627,0.995464,16,False
8,2015-05-09,0.396377,0.999496,16,False
9,2015-05-25,0.318383,0.805090,16,False



MISSING VALUES


,0
power_t2m_mean_c,0.00
power_precip_mm_day,0.00
smap_soil_moisture_am_m3m3,66.45
ndvi_mean,1.92
soil_ph,0.00
soil_clay_pct,0.00
gaez_whe_hilm_suitable_pct,0.00


In [24]:
# Composite quality and consistency check

composite_check = (
    df.dropna(subset=["ndvi_composite_date"])
      .groupby("ndvi_composite_date")
      .agg(
          observations=("ndvi_mean", "count"),
          ndvi_unique_values=("ndvi_mean", "nunique"),
          ndvi_min=("ndvi_mean", "min"),
          ndvi_max=("ndvi_mean", "max"),
          coverage=("ndvi_pixel_coverage_frac", "mean"),
          low_coverage=("ndvi_low_coverage_flag", "max"),
          first_daily_date=("date", "min"),
          last_daily_date=("date", "max")
      )
      .reset_index()
)

print("Total composites:", len(composite_check))

print("\nLow coverage composites:")
print(composite_check["low_coverage"].value_counts(dropna=False))

print("\nObservations per composite:")
print(composite_check["observations"].value_counts().sort_index())

print("\nNDVI variation within composites:")
print(
    composite_check["ndvi_unique_values"]
    .value_counts()
    .sort_index()
)

print("\nComposite date intervals:")
print(
    composite_check["ndvi_composite_date"]
    .sort_values()
    .diff()
    .value_counts()
)

display(composite_check.head(15))

Total composites: 248

Low coverage composites:
low_coverage
False    238
True      10
Name: count, dtype: int64

Observations per composite:
observations
13      7
14      3
16    238
Name: count, dtype: int64

NDVI variation within composites:
ndvi_unique_values
1    248
Name: count, dtype: int64

Composite date intervals:
ndvi_composite_date
16 days    237
13 days      7
14 days      3
Name: count, dtype: int64


,ndvi_composite_date,observations,ndvi_unique_values,ndvi_min,ndvi_max,coverage,low_coverage,first_daily_date,last_daily_date
0,2015-01-01,16,1,0.687144,0.687144,0.997984,False,2015-01-01,2015-01-16
1,2015-01-17,16,1,0.703984,0.703984,0.982361,False,2015-01-17,2015-02-01
2,2015-02-02,16,1,0.697882,0.697882,0.992944,False,2015-02-02,2015-02-17
3,2015-02-18,16,1,0.682736,0.682736,0.966234,False,2015-02-18,2015-03-05
4,2015-03-06,16,1,0.682831,0.682831,0.946201,False,2015-03-06,2015-03-21
5,2015-03-22,16,1,0.602478,0.602478,0.999748,False,2015-03-22,2015-04-06
6,2015-04-07,16,1,0.518043,0.518043,0.995464,False,2015-04-07,2015-04-22
7,2015-04-23,16,1,0.399627,0.399627,0.995464,False,2015-04-23,2015-05-08
8,2015-05-09,16,1,0.396377,0.396377,0.999496,False,2015-05-09,2015-05-24
9,2015-05-25,16,1,0.318383,0.318383,0.805090,False,2015-05-25,2015-06-09


In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/FarmWise_Unified_Dataset.csv")

df["date"] = pd.to_datetime(df["date"])
df["ndvi_composite_date"] = pd.to_datetime(
    df["ndvi_composite_date"], errors="coerce"
)

# Keep one record per NDVI composite
targets = (
    df.dropna(subset=["ndvi_composite_date", "ndvi_mean"])
      .groupby("ndvi_composite_date")
      .agg(
          ndvi_mean=("ndvi_mean", "first"),
          low_coverage=("ndvi_low_coverage_flag", "max"),
          coverage=("ndvi_pixel_coverage_frac", "mean"),
          observations=("ndvi_mean", "count")
      )
      .reset_index()
      .sort_values("ndvi_composite_date")
)

# Exclude low-coverage targets
targets = targets[targets["low_coverage"] == False].copy()

# Static features
static_cols = [
    c for c in df.columns
    if c.startswith("soil_") or c.startswith("gaez_")
]

static_data = df[static_cols].iloc[0]

weather_cols = [
    "power_t2m_mean_c",
    "power_t2m_max_c",
    "power_t2m_min_c",
    "power_precip_mm_day",
    "power_rh2m_pct",
    "power_wind2m_ms",
    "power_solar_rad_mj_m2_day",
    "gpm_precip_mm_day",
    "smap_soil_moisture_am_m3m3"
]

rows = []

for _, target in targets.iterrows():

    target_date = target["ndvi_composite_date"]

    # Previous 16 days only
    start_date = target_date - pd.Timedelta(days=16)
    end_date = target_date - pd.Timedelta(days=1)

    window = df[
        (df["date"] >= start_date) &
        (df["date"] <= end_date)
    ]

    if len(window) != 16:
        continue

    row = {
        "composite_date": target_date,
        "ndvi_mean": target["ndvi_mean"],
        "ndvi_coverage": target["coverage"],
        "previous_days_available": len(window)
    }

    for col in weather_cols:
        row[f"{col}_mean_prev16d"] = window[col].mean()

    row["smap_valid_days_prev16d"] = (
        window["smap_soil_moisture_am_m3m3"].notna().sum()
    )

    for col in static_cols:
        row[col] = static_data[col]

    row["month"] = target_date.month
    row["day_of_year"] = target_date.dayofyear

    rows.append(row)

ml_df = pd.DataFrame(rows)

output_path = "/content/FarmWise_ML_Dataset.csv"
ml_df.to_csv(output_path, index=False)

print("ML DATASET CREATED")
print("Shape:", ml_df.shape)
print("Date range:",
      ml_df["composite_date"].min(),
      "to",
      ml_df["composite_date"].max())

print("\nMissing values:")
display(
    ml_df.isna().mean().mul(100).round(2)
    .sort_values(ascending=False)
    .head(15)
)

print("\nPreview:")
display(ml_df.head())

print("\nSaved:", output_path)

ML DATASET CREATED
Shape: (237, 52)
Date range: 2015-01-17 00:00:00 to 2025-09-30 00:00:00

Missing values:


,0
smap_soil_moisture_am_m3m3_mean_prev16d,3.38
composite_date,0.00
ndvi_coverage,0.00
ndvi_mean,0.00
previous_days_available,0.00
power_t2m_mean_c_mean_prev16d,0.00
power_t2m_min_c_mean_prev16d,0.00
power_t2m_max_c_mean_prev16d,0.00
power_rh2m_pct_mean_prev16d,0.00
power_wind2m_ms_mean_prev16d,0.00



Preview:


,composite_date,ndvi_mean,ndvi_coverage,previous_days_available,power_t2m_mean_c_mean_prev16d,power_t2m_max_c_mean_prev16d,power_t2m_min_c_mean_prev16d,power_precip_mm_day_mean_prev16d,power_rh2m_pct_mean_prev16d,power_wind2m_ms_mean_prev16d,...,gaez_rcw_hilm_suitable_pct,gaez_rcw_hrlm_suitable_pct,gaez_rcw_lilm_suitable_pct,gaez_rcw_lrlm_suitable_pct,gaez_whe_hilm_suitable_pct,gaez_whe_hrlm_suitable_pct,gaez_whe_lilm_suitable_pct,gaez_whe_lrlm_suitable_pct,month,day_of_year
0,2015-01-17,0.703984,0.982361,16,11.197500,17.176250,6.180000,0.530000,60.857500,3.420625,...,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0,1,17
1,2015-02-02,0.697882,0.992944,16,14.105625,22.333125,8.064375,0.019375,52.108750,2.063125,...,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0,2,33
2,2015-02-18,0.682736,0.966234,16,13.215000,20.690625,7.458125,0.198125,53.610625,2.688125,...,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0,2,49
3,2015-03-06,0.682831,0.946201,16,14.118750,21.338750,8.504375,0.411250,61.163125,2.485000,...,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0,3,65
4,2015-03-22,0.602478,0.999748,16,17.033125,25.784375,10.277500,0.133125,59.218125,2.428125,...,90.91,0.0,90.91,0.0,90.28,0.0,90.28,0.0,3,81



Saved: /content/FarmWise_ML_Dataset.csv


In [26]:
import pandas as pd

ml_df = pd.read_csv("/content/FarmWise_ML_Dataset.csv")

ml_df["composite_date"] = pd.to_datetime(
    ml_df["composite_date"]
)

# Chronological split with 16-day gaps
train = ml_df[
    ml_df["composite_date"] < "2022-01-01"
].copy()

validation = ml_df[
    (ml_df["composite_date"] >= "2022-01-17") &
    (ml_df["composite_date"] < "2024-01-01")
].copy()

test = ml_df[
    ml_df["composite_date"] >= "2024-01-17"
].copy()

print("=" * 45)
print("CHRONOLOGICAL DATA SPLIT")
print("=" * 45)

for name, data in [
    ("TRAIN", train),
    ("VALIDATION", validation),
    ("TEST", test)
]:
    print(f"\n{name}")
    print("Rows:", len(data))
    if len(data):
        print("Start:", data["composite_date"].min())
        print("End:", data["composite_date"].max())

print("\nTotal used:", len(train) + len(validation) + len(test))
print("Total original:", len(ml_df))
print("Excluded in gaps:",
      len(ml_df) - len(train) - len(validation) - len(test))

# Save separate splits
train.to_csv("/content/FarmWise_Train.csv", index=False)
validation.to_csv("/content/FarmWise_Validation.csv", index=False)
test.to_csv("/content/FarmWise_Test.csv", index=False)

print("\nSplit files saved successfully!")

CHRONOLOGICAL DATA SPLIT

TRAIN
Rows: 153
Start: 2015-01-17 00:00:00
End: 2021-12-19 00:00:00

VALIDATION
Rows: 43
Start: 2022-01-17 00:00:00
End: 2023-12-19 00:00:00

TEST
Rows: 39
Start: 2024-01-17 00:00:00
End: 2025-09-30 00:00:00

Total used: 235
Total original: 237
Excluded in gaps: 2

Split files saved successfully!


In [27]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load splits
train = pd.read_csv("/content/FarmWise_Train.csv")
val = pd.read_csv("/content/FarmWise_Validation.csv")
test = pd.read_csv("/content/FarmWise_Test.csv")

# Target
target = "ndvi_mean"

# Columns that must not be model inputs
drop_cols = [
    "composite_date",
    "ndvi_mean",
    "ndvi_coverage",
    "previous_days_available"
]

# Prepare features
X_train = train.drop(columns=drop_cols)
X_val = val.drop(columns=drop_cols)
X_test = test.drop(columns=drop_cols)

y_train = train[target]
y_val = val[target]
y_test = test[target]

# Keep only numeric features
X_train = X_train.select_dtypes(include=np.number)
X_val = X_val[X_train.columns]
X_test = X_test[X_train.columns]

# Impute missing values using training data only
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

# Scale using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("PREPROCESSING COMPLETED")
print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Testing:", X_test_scaled.shape)
print("Features:", len(X_train.columns))
print("Missing values after imputation:",
      np.isnan(X_train_scaled).sum(),
      np.isnan(X_val_scaled).sum(),
      np.isnan(X_test_scaled).sum())

PREPROCESSING COMPLETED
Training: (153, 48)
Validation: (43, 48)
Testing: (39, 48)
Features: 48
Missing values after imputation: 0 0 0
